# 02 · Indexes: dictionary, postings, positions, zones, parameters

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
from scholar_ir.config import load_config, p
cfg = load_config()  # config.yaml at the repo root (or $SCHOLAR_IR_CONFIG)
R = p(cfg, "results_dir")

In [ ]:
from scholar_ir.engine import SearchEngine
engine = SearchEngine.load(cfg)
index, papers = engine.index, engine.papers
print(f"{index.N:,} papers, years {papers.year.min()}-{papers.year.max()}, tuned params loaded: {bool(engine.tuned)}")

In [ ]:
pd.read_csv(R / 'index/index_stats.csv')

In [ ]:
json.loads((R / 'index/build_info.json').read_text())

## Dictionary entry and postings of one term, per zone
Postings are `(doc_id, tf, [positions])`; the dictionary stores df and a pointer into the postings arrays.

In [ ]:
term = index.analyzer.terms('retrieval')[0]
for z in ('title', 'abstract', 'all'):
    zi = index.zone(z); t = zi.term_id(term)
    print(f'{z:8} term_id={t} df={zi.df[t]} pointer=[{zi.ptr[t]}, {zi.ptr[t+1]})  first postings: {zi.raw_postings(term, 3)}')

## Boolean, phrase and proximity queries (postings touched are counted)

In [ ]:
from scholar_ir.boolean_search import boolean_search
from scholar_ir.query_parser import parse
for q in ['"neural information retrieval"', 'retrieval AND augmented AND generation',
          '"query expansion" NEAR/5', 'title:survey AND recommender year:2015..2020', 'transformer NOT vision cat:cs.IR']:
    pq = parse(q); docs, st = boolean_search(index, pq)
    print(f'{q:55} -> {len(docs):5} docs, touched={st.touched:7}, lists={st.lists[:4]}')

## Query optimisation: process AND terms in increasing df

In [ ]:
pd.read_csv(R / 'efficiency/boolean_and_summary.csv')

## Effect of stemming and stop words on the vocabulary

In [ ]:
pd.read_csv(R / 'index/vocabulary_ablation.csv')

## Variable-byte gap compression of the doc-id postings

In [ ]:
from scholar_ir.index import vb_encode, vb_decode
d = index.zone('all').doc_list(term)[:8].tolist(); gaps = [d[0]] + [b - a for a, b in zip(d, d[1:])]
enc = vb_encode(gaps); print('doc ids', d, '\ngaps   ', gaps, '\nbytes  ', list(enc), '\ndecoded', vb_decode(enc))